In [1]:
import os
import glob
import polars as pl
import numpy as np
from pathlib import Path

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Raw data directory (needed to extract ground-truth 'round_end' winners)
RAW_DATA_DIR = Path(r"F:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds")

# Feature Data Directories (Inputs)
TRAIN_FEAT_DIR = Path(r"F:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Training\Mirage")
TEST_FEAT_DIR = Path(r"F:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Testing\Mirage")

# Results Directories (Outputs)
TRAIN_OUT_DIR = Path(r"F:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\Results\STOMs\WP_DAC_01\Training\Mirage")
TEST_OUT_DIR = Path(r"F:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\Results\STOMs\WP_DAC_01\Testing\Mirage")

EPOCHS = [0, 10, 20, 30, 40, 50, 60, "End"]
GRID_SIZE = 10

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# ==========================================
# 2. GROUND TRUTH RETRIEVAL
# ==========================================
def get_actual_winners(match_id):
    """Finds the raw round_end parquet file to extract ground truth results for the match."""
    match_dirs = list(RAW_DATA_DIR.rglob(match_id))
    if not match_dirs:
        return {}

    match_dir = match_dirs[0]
    re_files = glob.glob(str(match_dir / "round_end*"))
    
    if not re_files:
        return {}

    # Load and standardize winner team codes
    df_end = pl.read_parquet(re_files[0])
    team_map = {"2": "T", "3": "CT", "TERRORIST": "T", "CT": "CT", "T": "T"}
    
    df_end = df_end.with_columns(
        pl.col("winner_team_code").cast(pl.Utf8).replace(team_map).alias("actual_winner")
    ).select(["round", "actual_winner"])
    
    return dict(zip(df_end["round"].to_list(), df_end["actual_winner"].to_list()))

# ==========================================
# 3. CORE PROCESSING LOGIC
# ==========================================
def process_feature_directory(input_dir, output_dir):
    """Processes all match CSVs in the given input directory and saves the 4-row matrix."""
    
    # Ensure output directory exists
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Get all match feature files
    match_files = list(input_dir.glob("*.csv"))
    
    if not match_files:
        print(f"No CSV files found in: {input_dir}")
        return

    print(f"\nProcessing {len(match_files)} matches from {input_dir.parent.name}/{input_dir.name}...")

    for filepath in match_files:
        match_id = filepath.stem
        df_feat = pl.read_csv(filepath)
        actual_winners_map = get_actual_winners(match_id)
        
        rounds = df_feat["round"].to_list()
        
        # Tracking dictionaries for match-level aggregation per epoch
        ct_counts_agg = {str(e): [] for e in EPOCHS}
        t_counts_agg = {str(e): [] for e in EPOCHS}
        ct_probs_agg = {str(e): [] for e in EPOCHS}
        accuracy_agg = {str(e): {"correct": 0, "total": 0} for e in EPOCHS}

        # Iterate through every round in the match
        for r_idx in range(len(rounds)):
            round_data = df_feat.row(r_idx, named=True)
            r_id = round_data["round"]
            actual_winner = actual_winners_map.get(r_id, "Unknown")

            for epoch in EPOCHS:
                e_str = str(epoch)
                
                # Compute Spatially Collapsed Counts (Summing over cells with any activity)
                ct_occ = sum(1 for i in range(GRID_SIZE**2) if round_data.get(f"CT_e{epoch}_cell_{i:02d}", 0) > 0)
                t_occ = sum(1 for i in range(GRID_SIZE**2) if round_data.get(f"T_e{epoch}_cell_{i:02d}", 0) > 0)
                
                # Win Probability (Sigmoid of the difference)
                prob_ct = sigmoid(ct_occ - t_occ)
                
                # Store aggregated metrics
                ct_counts_agg[e_str].append(ct_occ)
                t_counts_agg[e_str].append(t_occ)
                ct_probs_agg[e_str].append(prob_ct)
                
                # Evaluation
                if actual_winner in ["CT", "T"]:
                    pred_winner = "CT" if prob_ct > 0.5 else "T"
                    accuracy_agg[e_str]["total"] += 1
                    if pred_winner == actual_winner:
                        accuracy_agg[e_str]["correct"] += 1

        # ==========================================
        # 4. BUILD THE 4-ROW MATRIX
        # ==========================================
        # Calculate match-level averages across all rounds for this match
        matrix_rows = {
            "Metric": [
                "Row 1: CT Aggregated Cells (Avg)", 
                "Row 2: T Aggregated Cells (Avg)", 
                "Row 3: CT Win Probability (Avg)", 
                "Row 4: Prediction Accuracy"
            ]
        }
        
        for e in EPOCHS:
            e_str = str(e)
            col_name = f"Epoch_{e_str}"
            
            # Averages across rounds
            ct_mean = np.mean(ct_counts_agg[e_str]) if ct_counts_agg[e_str] else 0
            t_mean = np.mean(t_counts_agg[e_str]) if t_counts_agg[e_str] else 0
            prob_mean = np.mean(ct_probs_agg[e_str]) if ct_probs_agg[e_str] else 0
            
            # Accuracy metric
            acc_data = accuracy_agg[e_str]
            accuracy = (acc_data["correct"] / acc_data["total"]) if acc_data["total"] > 0 else 0
            
            # Append column data
            matrix_rows[col_name] = [
                round(ct_mean, 2),
                round(t_mean, 2),
                round(prob_mean, 4),
                round(accuracy, 4)
            ]

        # Convert dictionary to Polars DataFrame and save
        df_matrix = pl.DataFrame(matrix_rows)
        out_filepath = output_dir / f"{match_id}.csv"
        df_matrix.write_csv(out_filepath)
        
    print(f"Finished processing and saving matrix files to: {output_dir}")

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # Step 1-5: Process Training Data
    process_feature_directory(TRAIN_FEAT_DIR, TRAIN_OUT_DIR)
    
    # Step 6: Repeat for Testing Data
    process_feature_directory(TEST_FEAT_DIR, TEST_OUT_DIR)
    
    print("\nAll match-specific results generated successfully.")


Processing 216 matches from Training/Mirage...
Finished processing and saving matrix files to: F:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\Results\STOMs\WP_DAC_01\Training\Mirage

Processing 54 matches from Testing/Mirage...
Finished processing and saving matrix files to: F:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\Results\STOMs\WP_DAC_01\Testing\Mirage

All match-specific results generated successfully.
